### Wine classification: data_split + scaler

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import math

In [4]:
# ===== 1. Wczytanie danych =====
file_path = os.path.join(os.getcwd(), 'wine.csv')
data = np.loadtxt(file_path, delimiter=',', dtype=np.float32, skiprows=1)
X = data[:, 1:]   # cechy
y = data[:, 0]    # klasy (1,2,3)

In [6]:
# ===== 2. Podział na zbiór treningowy i testowy =====
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)

In [8]:
# ===== 3. Skalowanie cech =====
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [10]:
# ===== 4. Przygotowanie Dataset =====
class WineDataset(Dataset):
    def __init__(self, X, y):
        self.x_data = torch.from_numpy(X)
        # odejmujemy 1, bo CrossEntropyLoss oczekuje klas 0,1,2
        self.y_data = torch.from_numpy(y).long() - 1
        self.n_samples = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.n_samples

train_dataset = WineDataset(X_train, y_train)
test_dataset = WineDataset(X_test, y_test)

train_loader = DataLoader(dataset=train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=4, shuffle=False)

In [12]:
# ===== 5. Definicja modelu =====
input_size = X_train.shape[1]
num_classes = 3
model = nn.Linear(input_size, num_classes)

In [14]:
# ===== 6. Loss + optimizer =====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [16]:
# ===== 7. Pętla treningowa =====
num_epochs = 10
total_samples = len(train_dataset)
n_iterations = math.ceil(total_samples / 4)

for epoch in range(num_epochs):
    for i, (inputs, labels) in enumerate(train_loader):
        # forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 5 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}, Step {i+1}/{n_iterations}, Loss = {loss.item():.4f}')

Epoch 1/10, Step 5/36, Loss = 0.8328
Epoch 1/10, Step 10/36, Loss = 0.6105
Epoch 1/10, Step 15/36, Loss = 0.5012
Epoch 1/10, Step 20/36, Loss = 0.7426
Epoch 1/10, Step 25/36, Loss = 0.1749
Epoch 1/10, Step 30/36, Loss = 0.3549
Epoch 1/10, Step 35/36, Loss = 0.3166
Epoch 2/10, Step 5/36, Loss = 0.1495
Epoch 2/10, Step 10/36, Loss = 0.2117
Epoch 2/10, Step 15/36, Loss = 0.0617
Epoch 2/10, Step 20/36, Loss = 0.0617
Epoch 2/10, Step 25/36, Loss = 0.1313
Epoch 2/10, Step 30/36, Loss = 0.1136
Epoch 2/10, Step 35/36, Loss = 0.1166
Epoch 3/10, Step 5/36, Loss = 0.1856
Epoch 3/10, Step 10/36, Loss = 0.0703
Epoch 3/10, Step 15/36, Loss = 0.6079
Epoch 3/10, Step 20/36, Loss = 0.0771
Epoch 3/10, Step 25/36, Loss = 0.2432
Epoch 3/10, Step 30/36, Loss = 0.2532
Epoch 3/10, Step 35/36, Loss = 0.0171
Epoch 4/10, Step 5/36, Loss = 0.0345
Epoch 4/10, Step 10/36, Loss = 0.1057
Epoch 4/10, Step 15/36, Loss = 0.0744
Epoch 4/10, Step 20/36, Loss = 0.1813
Epoch 4/10, Step 25/36, Loss = 0.0403
Epoch 4/10, Step

In [18]:
# ===== 8. Ocena dokładności na zbiorze testowym =====
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy on test set: {100 * correct / total:.2f}%')

Accuracy on test set: 97.22%
